[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/06_series/06_4_Text_in_a_Series.ipynb)

# 06.4: Text in a Series

You already know string methods such as `.lower()` and `.split()` from module 05, and they work on one string at a time. A series can hold thousands of strings, and you do not want to write a loop every time you need to search or clean them. This notebook introduces the `.str` accessor, which applies a string method to every value in a series.

## The data

`names` is a series holding the major names. Its index is also the major names, so that a mask made from `names` lines up with `salaries`.

In [ ]:
import pandas as pd

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
major_list = majors["Major"].to_list()
salaries = pd.Series(majors["Median"].to_list(), index=major_list)
names = pd.Series(major_list, index=major_list)
names.head(3)

## The `.str` accessor

A series does not have a `.lower()` method of its own, because `.lower()` is something that strings do. The next cell fails on purpose. If you use Run all, Colab stops at this cell, so run the cells after it yourself.

In [ ]:
names.lower()

Putting `.str` before the method tells pandas to use the string method on each value, which replaces the loop you would otherwise write. The result is a new series with the same labels.

In [ ]:
names.str.lower().head(3)

The values are now lower case, for example `petroleum engineering`, and the labels did not change. `.str.len()` gives the length of each string, which is how you can find the longest name.

In [ ]:
names.str.len().sort_values().tail(2)

The longest name has 65 characters: ELECTRICAL, MECHANICAL, AND PRECISION TECHNOLOGIES AND PRODUCTION.

## Searching: `.str.contains()`

`.str.contains()` makes a boolean mask that is `True` for every value that includes a piece of text. Everything you learned about masks in 06.2 applies, so you can count with it and filter with it.

In [ ]:
print(names.str.contains("ENGINEERING").sum())
names[names.str.contains("PSYCHOLOGY")]

Twenty-six major names include ENGINEERING. The list for PSYCHOLOGY has 8 majors, and one of them is COGNITIVE SCIENCE AND BIOPSYCHOLOGY, which matches because `.str.contains()` looks for the text anywhere, including inside a longer word. Matching is also case-sensitive, so searching for lower case text finds nothing in this data, and pandas gives no warning about it.

In [ ]:
names.str.contains("engineering").sum()

That result of 0 is a common way to get a wrong answer without an error. Here is what a mask from text is good for. Since `names` and `salaries` have the same index, the mask can select from `salaries`, and we can compare groups.

In [ ]:
is_engineering = names.str.contains("ENGINEERING")
print(salaries[is_engineering].median())
print(salaries[~is_engineering].median())

The median for majors with ENGINEERING in the name is 57050, and for all other majors it is 35000. That is a real question about the data, answered with a search through text and one statistic.

## Starts with and ends with

`.str.startswith()` and `.str.endswith()` also produce masks, and they are stricter than `.str.contains()` because the text has to be in a specific place.

In [ ]:
names[names.str.startswith("GENERAL")]

Six majors start with GENERAL, including GENERAL ENGINEERING and GENERAL BUSINESS. By contrast, a name that only has GENERAL in the middle would not match.

## Replacing text

`.str.replace()` swaps one piece of text for another in every value. As with the operations in 06.2, the result is a new series unless you assign it.

In [ ]:
names.str.replace(" AND ", " & ").head(4)

Where a name had AND between spaces, such as MINING AND MINERAL ENGINEERING, it now has an ampersand. Names without it are unchanged.

## Splitting text

`.str.split()` breaks each string into a list of pieces, and `.str[0]` then takes the first piece of each list. The `.str` is needed again because each value is now a list. `.str[-1]` takes the last piece, using the same negative indexing as a list.

In [ ]:
first_word = names.str.split(" ").str[0]
first_word.value_counts().head(3)

The most common first word is MISCELLANEOUS with 10 majors, which suggests that this data has a lot of catch-all majors. The pattern of split, take a piece, then `.value_counts()` is a very common way to pull information out of text, as you will see in the pair activity.

## Cleaning up text

Text typed by people often has stray spaces and inconsistent capitals. To Python, each of these values is a different string, so they would be counted separately.

In [ ]:
answers = pd.Series(["Yes", "yes ", " YES", "No", "no"])
answers.value_counts()

Each of the five answers is counted as its own category, even though there are really two answers. `.str.strip()` removes spaces from the ends of each string. To apply two methods in a row, chain them, with `.str` before each one.

In [ ]:
answers.str.strip().str.lower().value_counts()

Now `yes` has a count of 3 and `no` has a count of 2.

## What's next

That finishes module 06. You can now create a series, look up and filter its values, summarize it and work with text in it. Real datasets have many columns, though, and a question usually involves more than one of them. Module 07 starts with notebook 07.1, which introduces the DataFrame, a table made of series that share an index.